# 店名补全（已完成，归档）

# 生成缺失店名商家的评论数据

本 Notebook 只做一件事：找出 `restaurants.csv` 中 `name` 为空的 `restId`，并从 `ratings.csv` 中筛出对应的非空文字评论，生成 `rating_miss_name.csv`。

原始两个 CSV 不会被修改。

In [172]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

PROJECT_DIR = Path(r"D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DERIVED_DIR = PROJECT_DIR / "data" / "derived"
FINAL_DIR = PROJECT_DIR / "data" / "final"
RATING_SPLIT_DIR = PROJECT_DIR / "data" / "splits" / "rating_split"
NAME_OUTPUT_DIR = PROJECT_DIR / "outputs" / "name_completion"
CATEGORY_OUTPUT_DIR = PROJECT_DIR / "outputs" / "category"
RESTAURANTS_PATH = RAW_DIR / 'restaurants.csv'
RATINGS_PATH = RAW_DIR / 'ratings.csv'
OUTPUT_PATH = DERIVED_DIR / 'rating_miss_name.csv'

assert RESTAURANTS_PATH.exists(), RESTAURANTS_PATH
assert RATINGS_PATH.exists(), RATINGS_PATH

## 数据概览

先确认文件规模、字段和编码是否正常，再开始筛选。评论文件很大，因此此处只读取表头和一条样例，不会加载全部评论。

In [173]:
def file_size_mb(path):
    return round(path.stat().st_size / 1024 / 1024, 2)

with RESTAURANTS_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    restaurant_reader = csv.DictReader(source)
    restaurant_columns = restaurant_reader.fieldnames
    restaurant_preview = [next(restaurant_reader) for _ in range(3)]

with RATINGS_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    rating_reader = csv.DictReader(source)
    rating_columns = rating_reader.fieldnames
    first_comment = next(row['comment'] for row in rating_reader if (row.get('comment') or '').strip())

print(f'restaurants.csv：{file_size_mb(RESTAURANTS_PATH):,} MB')
print('商家表字段：', restaurant_columns)
print('商家表前 3 条：', restaurant_preview)
print()
print(f'ratings.csv：{file_size_mb(RATINGS_PATH):,} MB')
print('评论表字段：', rating_columns)
print('评论样例：', first_comment[:120])

restaurants.csv：5.78 MB
商家表字段： ['restId', 'name']
商家表前 3 条： [{'restId': '0', 'name': ''}, {'restId': '1', 'name': ''}, {'restId': '2', 'name': ''}]

ratings.csv：1,790.06 MB
评论表字段： ['userId', 'restId', 'rating', 'rating_env', 'rating_flavor', 'rating_service', 'timestamp', 'comment']
评论样例： 经常去的，不过我自己的卡很久不用，被冻了，只能用爸爸的。吉利莲的巧克力以前选择多些，最近一次去只有一种选择了。食品零食的选择还不够。买不全。酸奶的日期都比较新鲜，逢年过节，时令食品的选择比较丰富。冷冻产品丰富，看看都不错。建议自己带好购物袋


## 第 1 步：找出缺失名称的 restId

In [174]:
missing_ids = set()
total_restaurants = 0

with RESTAURANTS_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    for row in reader:
        total_restaurants += 1
        if not (row.get('name') or '').strip():
            missing_ids.add(int(row['restId']))

print(f'商家总数：{total_restaurants:,}')
print(f'缺失名称的 restId 数：{len(missing_ids):,}')

商家总数：243,247
缺失名称的 restId 数：34,115


## 第 2 步：流式筛选评论并生成新 CSV

逐条读取完整评论文件，不会把大文件一次性放进内存。仅保留 `comment` 非空的记录，因为纯评分无法用于判断店名。输出使用 UTF-8 标记，Excel 可正确识别中文。

In [175]:
kept_rows = 0
total_rating_rows = 0

with RATINGS_PATH.open('r', encoding='utf-8-sig', newline='') as source, \
     OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    reader = csv.DictReader(source)
    input_columns = reader.fieldnames
    writer = csv.DictWriter(target, fieldnames=input_columns)
    writer.writeheader()

    for row in reader:
        total_rating_rows += 1
        comment = (row.get('comment') or '').strip()
        if int(row['restId']) in missing_ids and comment:
            writer.writerow(row)
            kept_rows += 1

print(f'已生成：{OUTPUT_PATH.name}')
print(f'筛选前（ratings.csv）：{total_rating_rows:,} 行 × {len(input_columns)} 列')
print(f'筛选后（rating_miss_name.csv）：{kept_rows:,} 行 × {len(input_columns)} 列')
print(f'文件位置：{OUTPUT_PATH}')

已生成：rating_miss_name.csv
筛选前（ratings.csv）：4,422,473 行 × 8 列
筛选后（rating_miss_name.csv）：744,124 行 × 8 列
文件位置：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\rating_miss_name.csv


## 第 3 步：检查筛选后评论数据的缺失值

这里将空字符串和空值都视为缺失。特别注意：`comment` 为空表示该评分没有文字评论，并不代表整条评分记录无效。

In [176]:
missing_counts = {}
total_rows = 0

with OUTPUT_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    missing_counts = {column: 0 for column in reader.fieldnames}

    for row in reader:
        total_rows += 1
        for column in reader.fieldnames:
            if not (row.get(column) or '').strip():
                missing_counts[column] += 1

print(f'检查行数：{total_rows:,}')
print('\n各列缺失值情况：')
for column, count in missing_counts.items():
    print(f'{column:<16} 缺失 {count:>9,} 条，占比 {count / total_rows:.2%}')

检查行数：744,124

各列缺失值情况：
userId           缺失         0 条，占比 0.00%
restId           缺失         0 条，占比 0.00%
rating           缺失   114,661 条，占比 15.41%
rating_env       缺失   297,432 条，占比 39.97%
rating_flavor    缺失   281,901 条，占比 37.88%
rating_service   缺失   297,432 条，占比 39.97%
timestamp        缺失         0 条，占比 0.00%
comment          缺失         0 条，占比 0.00%


## 第 4 步：逐条评论提取店名候选

不使用已有店名直接匹配。仅从当前评论中的“XX火锅、XX超市、去XX吃饭、在XX购物”等表达提取候选；“这家火锅、附近超市”等泛称会被排除。

In [177]:
# 含商家业态后缀的名称候选，例如：小龙坎火锅、优之良品超市。
SUFFIX_PATTERN = re.compile(
    r'([\u4e00-\u9fffA-Za-z0-9·]{2,16}(?:火锅|餐厅|饭店|酒家|咖啡|超市|便利店|酒吧|烤肉|烘焙|甜品|奶茶|茶饮|小吃店|快餐店))'
)

# 动作 + 商家 + 消费行为，例如：去星巴克喝咖啡、在麦德龙购物。
ACTION_PATTERN = re.compile(
    r'(?:在|去|到|来)([\u4e00-\u9fffA-Za-z0-9·]{2,14}?)(?:吃饭|吃|喝|买|购物|用餐|点餐|住|消费)'
)

GENERIC_WORDS = {
    '这家', '这里', '那家', '那个', '本店', '店里', '附近', '商场', '饭店', '餐厅',
    '超市', '便利店', '火锅', '咖啡', '购物', '吃饭', '地方', '他们家', '我们家',
    '上面', '淘宝上', '京东上', '当当上', '网上', '网站'
}

def clean_candidate(candidate):
    candidate = candidate.strip(' ，。！？、;；:：()（）')
    candidate = re.sub(r'^(?:在|去|到|来|从)', '', candidate)
    for action_word in ('购物', '用餐', '点餐', '消费', '吃饭', '吃', '喝', '买', '住'):
        position = candidate.find(action_word)
        if position >= 2:
            candidate = candidate[:position]
    if len(candidate) < 2 or candidate in GENERIC_WORDS:
        return ''
    if any(word in candidate for word in ('这家', '那家', '附近', '这里', '一个', '一家')):
        return ''
    if candidate.endswith('上') and candidate[:-1] in {'淘宝', '京东', '当当', '苏宁'}:
        return ''
    return candidate

def extract_name_candidates(comment):
    candidates = set()
    for raw_candidate in SUFFIX_PATTERN.findall(comment):
        candidate = clean_candidate(raw_candidate)
        if candidate:
            candidates.add(candidate)
    for raw_candidate in ACTION_PATTERN.findall(comment):
        candidate = clean_candidate(raw_candidate)
        if candidate:
            candidates.add(candidate)
    return candidates

demo_comments = [
    '小龙坎火锅的牛肉很好吃。',
    '去星巴克喝咖啡很方便。',
    '这家火锅服务不错。',
]
for text in demo_comments:
    print(text, '->', extract_name_candidates(text))

小龙坎火锅的牛肉很好吃。 -> {'小龙坎火锅'}
去星巴克喝咖啡很方便。 -> {'星巴克'}
这家火锅服务不错。 -> set()


## 第 5 步：按 restId 聚合候选票数

同一条评论对同一候选最多投一票。输出仅包含 `restId`、候选名称和票数，不会自动写回商家名称。

In [178]:
name_votes = defaultdict(Counter)

with OUTPUT_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    for row in reader:
        rest_id = int(row['restId'])
        candidates = extract_name_candidates(row['comment'])
        name_votes[rest_id].update(candidates)

candidate_rows = [
    {'restId': rest_id, 'name_candidate': candidate, 'vote_count': vote_count}
    for rest_id, votes in name_votes.items()
    for candidate, vote_count in votes.most_common()
]
CANDIDATE_OUTPUT_PATH = NAME_OUTPUT_DIR / 'name_candidate_votes.csv'
with CANDIDATE_OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=['restId', 'name_candidate', 'vote_count'])
    writer.writeheader()
    writer.writerows(candidate_rows)



## 第 6 步：清洗泛称、位置词和平台别名

该步骤会排除“他家、那里、这个超市、负一楼的超市、地下超市、里面、这边”等非店名表达；将“卓越上”等平台别名规范化；平台名称保留为“平台/线上商家候选”，参与后续投票。

In [179]:
GENERIC_EXACT = {
    '他家', '她家', '他们家', '我们家', '那里', '这里', '这儿', '那儿', '里面', '这边', '那边',
    '第一次', '第二次', '以前', '后来', '最近',
    '上面', '网上', '平台上', '网络上',
    '这个超市', '这家超市', '地下超市', '负一楼的超市', '负一层的超市',
    '超市', '便利店', '火锅', '餐厅', '饭店', '咖啡', '购物', '吃饭', '地方', '丝袜奶茶'
}
GENERIC_CONTAINS = (
    '这家', '那家', '这个', '那个', '这里', '那里', '里面', '这边', '那边',
    '附近', '地下', '负一楼', '负一层', '楼下', '楼上的', '一楼的', '二楼的', '开在'
)
FLOOR_DESCRIPTION_PATTERN = re.compile(
    r'^(?:负?[一二三四五六七八九十0-9]+(?:层|楼)|地下[一二三四五六七八九十0-9]*(?:层|楼)?)(?:是|有)'
)

FLOOR_STORE_PATTERN = re.compile(
    r'^(?:\u8d1f?[\u4e00-\u9fff0-9]+(?:\u5c42|\u697c)|\u5730\u4e0b[\u4e00-\u9fff0-9]*(?:\u5c42|\u697c)?)(?:\u7684)?(?:\u8d85\u5e02|\u706b\u9505|\u9910\u5385|\u996d\u5e97|\u4fbf\u5229\u5e97)$'
)
HISTORICAL_PREFIXES = ('原来的', '原来', '旧址的', '旧的')

PLATFORM_ALIASES = {
    '卓越上': '卓越网', '卓越': '卓越网', '卓越网': '卓越网',
    '淘宝上': '淘宝', '淘宝网上': '淘宝', '淘宝网': '淘宝', 'TB': '淘宝',
    '京东上': '京东', '当当上': '当当', '一号店': '1号店',
    '易迅上': '易迅网', '易迅': '易迅网', '易迅网': '易迅网',
    '凡客上': '凡客诚品', '凡客': '凡客诚品', '凡客诚品': '凡客诚品',
    '可得网上': '可得网', '可得网': '可得网', '可得': '可得网'
}
MERCHANT_ALIASES = {'全家便利店': '全家'}
PLATFORM_NAMES = {'淘宝', '京东', '卓越网', '当当', '苏宁', '1号店', '易迅网', '凡客诚品', '可得网'}
PLATFORM_TERMS = {
    '淘宝': ('淘宝', '淘宝网'),
    '京东': ('京东', '京东商城'),
    '卓越网': ('卓越网', '卓越亚马逊', '卓越'),
    '当当': ('当当网', '当当'),
    '苏宁': ('苏宁易购', '苏宁'),
    '1号店': ('1号店', '一号店'),
    '易迅网': ('易迅网', '易迅'),
    '凡客诚品': ('凡客诚品', '凡客'),
    '可得网': ('可得网', '可得'),
}

def clean_and_label_candidate(raw_candidate):
    candidate = raw_candidate.strip(' ，。！？、;；:：()（）')
    candidate = re.sub(r'^(?:在|去|到|来|从)', '', candidate)
    for action_word in ('购物', '用餐', '点餐', '消费', '吃饭', '吃', '喝', '买', '住'):
        position = candidate.find(action_word)
        if position >= 2:
            candidate = candidate[:position]
    candidate = candidate.strip()

    if candidate.isascii() and any(char.isalpha() for char in candidate):
        return None
    if len(candidate) < 2 or candidate in GENERIC_EXACT:
        return None
    if any(word in candidate for word in GENERIC_CONTAINS):
        return None
    if FLOOR_DESCRIPTION_PATTERN.match(candidate) or FLOOR_STORE_PATTERN.match(candidate):
        return None
    if candidate.startswith(HISTORICAL_PREFIXES):
        return None

    normalized = PLATFORM_ALIASES.get(candidate, candidate)
    normalized = MERCHANT_ALIASES.get(normalized, normalized)
    status = '平台/线上商家候选' if normalized in PLATFORM_NAMES else '商家名称候选'
    return normalized, status

EXPLICIT_ENGLISH_NAME_PATTERN = re.compile(
    r'(?:叫|名为|叫做|叫作)\s*([A-Za-z][A-Za-z .&\'-]{2,40})'
)

# 也识别“是 Denny House”“看到 Denny House”等明确的英文店名表达。
EXPLICIT_ENGLISH_NAME_PATTERN = re.compile(
    r'(?:\u53eb|\u540d\u4e3a|\u53eb\u505a|\u53eb\u4f5c|\u662f|\u770b\u5230|\u770b\u89c1)\s*([A-Za-z][A-Za-z &\'-]{2,40})'
)

# 收紧英文规则：普通英文名称需在明确命名语境中；“是 + 英文名”仅接受全大写名称。
EXPLICIT_ENGLISH_NAME_PATTERN = re.compile(
    r'(?:\u53eb|\u540d\u4e3a|\u53eb\u505a|\u53eb\u4f5c|\u770b\u5230|\u770b\u89c1)\s*([A-Za-z][A-Za-z &\'-]{2,40})'
)
EXPLICIT_UPPERCASE_NAME_PATTERN = re.compile(
    r'\u662f\s*([A-Z][A-Z &\'-]{2,40})'
)

def extract_platform_mentions(comment):
    return {platform for platform, terms in PLATFORM_TERMS.items()
            if any(term in comment for term in terms)}

def extract_clean_candidates(comment):
    extracted = {}
    raw_candidates = SUFFIX_PATTERN.findall(comment) + ACTION_PATTERN.findall(comment)
    for raw_candidate in raw_candidates:
        result = clean_and_label_candidate(raw_candidate)
        if result is not None:
            candidate, status = result
            if candidate.isascii() and any(char.isalpha() for char in candidate):
                candidate = ' '.join(word.capitalize() for word in candidate.split())
                # 单个英文词更常是人名、商品品牌或普通词，不作为店名候选。
                if len(candidate.split()) < 2:
                    continue
            extracted[candidate] = status
    return extracted

for text in ['负一楼的超市东西很全', '去卓越上买书', '在淘宝网上买东西', '小龙坎火锅很好吃']:
    print(text, '->', extract_clean_candidates(text))

负一楼的超市东西很全 -> {}
去卓越上买书 -> {'卓越网': '平台/线上商家候选'}
在淘宝网上买东西 -> {'淘宝': '平台/线上商家候选'}
小龙坎火锅很好吃 -> {'小龙坎火锅': '商家名称候选'}


In [180]:
name_votes = defaultdict(Counter)
platform_votes = defaultdict(Counter)
candidate_status = {}

with OUTPUT_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    for row in reader:
        rest_id = int(row['restId'])
        comment = row['comment']
        for platform in extract_platform_mentions(comment):
            platform_votes[rest_id][platform] += 1
        for candidate, status in extract_clean_candidates(comment).items():
            name_votes[rest_id][candidate] += 1
            candidate_status[candidate] = status

candidate_rows = [
    {
        'restId': rest_id,
        'name_candidate': candidate,
        'candidate_status': candidate_status[candidate],
        'vote_count': vote_count,
    }
    for rest_id, votes in name_votes.items()
    for candidate, vote_count in votes.most_common()
]
CANDIDATE_OUTPUT_PATH = NAME_OUTPUT_DIR / 'name_candidate_votes.csv'
with CANDIDATE_OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(
        target, fieldnames=['restId', 'name_candidate', 'candidate_status', 'vote_count']
    )
    writer.writeheader()
    writer.writerows(candidate_rows)

platform_rows = [
    {'restId': rest_id, 'platform_name': platform, 'vote_count': vote_count}
    for rest_id, votes in platform_votes.items()
    for platform, vote_count in votes.most_common()
]
PLATFORM_OUTPUT_PATH = NAME_OUTPUT_DIR / 'platform_reference_votes.csv'
with PLATFORM_OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=['restId', 'platform_name', 'vote_count'])
    writer.writeheader()
    writer.writerows(platform_rows)

print(f'清洗后识别出候选的 restId 数：{len(name_votes):,}')
print(f'清洗后候选记录数：{len(candidate_rows):,}')
print(f'候选结果文件：{CANDIDATE_OUTPUT_PATH}')
print(f'平台词票数文件：{PLATFORM_OUTPUT_PATH}')

清洗后识别出候选的 restId 数：17,779
清洗后候选记录数：139,408
候选结果文件：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\name_candidate_votes.csv
平台词票数文件：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\platform_reference_votes.csv


## 第 7 步：保留高置信度候选

候选名称仅出现 1 次时，人工检查发现包含大量泛称、句子片段和误提取词。该步骤不删除原始候选表，而是另存票数不少于 2 的高价值候选，用于后续审核和店名补全。

In [181]:
MIN_HIGH_VALUE_VOTES = 2
ACCEPTED_CANDIDATES_PATH = NAME_OUTPUT_DIR / 'name_candidates_ge3_votes.csv'
TWO_VOTE_CANDIDATES_PATH = NAME_OUTPUT_DIR / 'name_two_vote_candidates.csv'
HIGH_VALUE_CANDIDATES_PATH = NAME_OUTPUT_DIR / 'name_high_value_candidates.csv'

high_value_candidates = sorted(
    (row for row in candidate_rows if int(row['vote_count']) >= MIN_HIGH_VALUE_VOTES),
    key=lambda row: (-int(row['vote_count']), int(row['restId']), row['name_candidate'])
)
accepted_candidates = [row for row in high_value_candidates if int(row['vote_count']) >= 3]
two_vote_candidates = [row for row in high_value_candidates if int(row['vote_count']) == 2]

def write_candidate_csv(path, rows):
    with path.open('w', encoding='utf-8-sig', newline='') as target:
        writer = csv.DictWriter(
            target,
            fieldnames=['restId', 'name_candidate', 'candidate_status', 'vote_count']
        )
        writer.writeheader()
        writer.writerows(rows)

write_candidate_csv(HIGH_VALUE_CANDIDATES_PATH, high_value_candidates)
write_candidate_csv(ACCEPTED_CANDIDATES_PATH, accepted_candidates)
write_candidate_csv(TWO_VOTE_CANDIDATES_PATH, two_vote_candidates)

print(f'原始候选记录数：{len(candidate_rows):,}')
print(f'已接受（票数不少于 3）的候选数：{len(accepted_candidates):,}')
print(f'待后续处理（票数恰好为 2）的候选数：{len(two_vote_candidates):,}')
print(f'已接受候选文件：{ACCEPTED_CANDIDATES_PATH}')
print(f'两票候选文件：{TWO_VOTE_CANDIDATES_PATH}')


原始候选记录数：139,408
已接受（票数不少于 3）的候选数：760
待后续处理（票数恰好为 2）的候选数：2,295
已接受候选文件：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\name_candidates_ge3_votes.csv
两票候选文件：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\name_two_vote_candidates.csv


## 第 8 步：合并已接受候选并生成补名草稿

合并票数不少于 3 的候选与大语言模型接受的两票候选。若同一 restId 有多个已接受候选，选择最高票名称；最高票并列时按候选排序取第一项。原始 `restaurants.csv` 不会被修改。

In [182]:
LLM_SCREENED_PATH = NAME_OUTPUT_DIR / 'name_two_vote_llm_screened.csv'
GE3_CANDIDATES_PATH = NAME_OUTPUT_DIR / 'name_candidates_ge3_votes.csv'
SELECTED_CANDIDATES_PATH = NAME_OUTPUT_DIR / 'name_completion_selected_candidates.csv'
FILL_EVIDENCE_PATH = NAME_OUTPUT_DIR / 'name_completion_fill_evidence.csv'
CONFLICT_PATH = NAME_OUTPUT_DIR / 'name_completion_name_conflicts.csv'
DRAFT_PATH = DERIVED_DIR / 'restaurants_name_draft_selected_candidates.csv'
ACCEPTED_LLM_LABELS = {'具体店名', '平台/线上商家名'}

with GE3_CANDIDATES_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    ge3_rows = list(csv.DictReader(source))
with LLM_SCREENED_PATH.open('r', encoding='utf-8-sig', newline='') as source:
    llm_rows = list(csv.DictReader(source))

selected_rows = []
for row in ge3_rows:
    selected_rows.append({
        'restId': row['restId'], 'name_candidate': row['name_candidate'],
        'candidate_status': row['candidate_status'], 'vote_count': int(row['vote_count']),
        'selection_source': '票数不少于3',
    })
for row in llm_rows:
    if row['llm_label'] in ACCEPTED_LLM_LABELS:
        selected_rows.append({
            'restId': row['restId'], 'name_candidate': row['name_candidate'],
            'candidate_status': row['candidate_status'], 'vote_count': int(row['vote_count']),
            'selection_source': f"大语言模型：{row['llm_label']}",
        })

selected_rows.sort(key=lambda row: (int(row['restId']), -row['vote_count'], row['name_candidate']))
selected_fields = ['restId', 'name_candidate', 'candidate_status', 'vote_count', 'selection_source']
def write_csv(path, rows, fieldnames):
    with path.open('w', encoding='utf-8-sig', newline='') as target:
        writer = csv.DictWriter(target, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(SELECTED_CANDIDATES_PATH, selected_rows, selected_fields)

candidates_by_id = defaultdict(list)
for row in selected_rows:
    candidates_by_id[row['restId']].append(row)

fill_rows = []
conflict_rows = []
for rest_id, rows in candidates_by_id.items():
    max_votes = max(row['vote_count'] for row in rows)
    leaders = [row for row in rows if row['vote_count'] == max_votes]
    summary = '；'.join(
        f"{row['name_candidate']}（{row['vote_count']}票，{row['selection_source']}）"
        for row in rows
    )
    if len(leaders) == 1:
        winner = leaders[0]
        fill_rows.append({
            **winner,
            'selected_rule': '已接受候选中票数唯一最高',
        })
        if len(rows) > 1:
            conflict_rows.append({
                'restId': rest_id, 'all_selected_candidates': summary,
                'resolution': f"已填补：{winner['name_candidate']}（唯一最高票）",
            })
    else:
        winner = leaders[0]
        fill_rows.append({
            **winner,
            'selected_rule': '最高票并列，按排序第一项选择',
        })
        conflict_rows.append({
            'restId': rest_id, 'all_selected_candidates': summary,
            'resolution': f"已填补：{winner['name_candidate']}（最高票并列，取第一项）",
        })

fill_rows.sort(key=lambda row: int(row['restId']))
write_csv(
    FILL_EVIDENCE_PATH, fill_rows,
    selected_fields + ['selected_rule']
)
write_csv(CONFLICT_PATH, conflict_rows, ['restId', 'all_selected_candidates', 'resolution'])

fill_map = {row['restId']: row['name_candidate'] for row in fill_rows}
with RESTAURANTS_PATH.open('r', encoding='utf-8-sig', newline='') as source, \
     DRAFT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    reader = csv.DictReader(source)
    output_fields = reader.fieldnames + ['name_fill_status']
    writer = csv.DictWriter(target, fieldnames=output_fields)
    writer.writeheader()
    for restaurant in reader:
        rest_id = restaurant['restId']
        if not (restaurant.get('name') or '').strip() and rest_id in fill_map:
            restaurant['name'] = fill_map[rest_id]
            restaurant['name_fill_status'] = 'selected_candidate_unique_top'
        else:
            restaurant['name_fill_status'] = ''
        writer.writerow(restaurant)

print(f'合并后的已接受候选记录数：{len(selected_rows):,}')
print(f'涉及的 restId 数：{len(candidates_by_id):,}')
print(f'已写入草稿的 restId 数：{len(fill_rows):,}')
tie_resolved_count = sum('最高票并列，取第一项' in row['resolution'] for row in conflict_rows)
print(f'按第一项解决的并列 restId 数：{tie_resolved_count:,}')
print(f'餐馆草稿：{DRAFT_PATH}')


合并后的已接受候选记录数：1,515
涉及的 restId 数：1,267
已写入草稿的 restId 数：1,267
按第一项解决的并列 restId 数：57
餐馆草稿：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\restaurants_name_draft_selected_candidates.csv


## 阶段成果（归档）

本阶段已完成候选提取、投票筛选和两票候选的大语言模型语义筛选。最终按已接受候选的最高票名称填补 1,267 个 restId；其中 57 个最高票并列的 restId 按候选排序取第一项。

后续类别分类应使用 `restaurants_name_draft_selected_candidates.csv` 作为基础表；本 Notebook 不再需要继续扩展。